In [93]:
# ==================================================
# Configuració del projecte
# ==================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [95]:
# ==================================================
# Imports
# ==================================================

import pandas as pd

from src.config import (
    DATA_PROCESSED,
)

from src.utils_io import (
    load_csv,
    save_csv,
)

from src.master import (
    merge_master,
)

In [97]:
# ==================================================
# Carregar datasets harmonitzats
# ==================================================

print("Carregant datasets harmonitzats...")

df_rent = load_csv(
    DATA_PROCESSED / "rent_harmonized.csv"
)

df_income = load_csv(
    DATA_PROCESSED / "income_harmonized.csv"
)

df_nat = load_csv(
    DATA_PROCESSED / "nationality_harmonized.csv"
)

df_pop = load_csv(
    DATA_PROCESSED / "population_harmonized.csv"
)

df_atur = load_csv(
    DATA_PROCESSED / "atur_harmonized.csv"
)

df_est = load_csv(
    DATA_PROCESSED / "estudis_harmonized.csv"
)

df_cont = load_csv(
    DATA_PROCESSED / "contractes_harmonized.csv"
)

df_sup = load_csv(
    DATA_PROCESSED / "superficie_harmonized.csv"
)

df_hut = load_csv(
    DATA_PROCESSED / "hut_harmonized.csv"
)

print("Datasets carregats correctament.")

Carregant datasets harmonitzats...
Datasets carregats correctament.


In [99]:
# ==================================================
# Funció auxiliar MERGE
# ==================================================

MERGE_KEYS = [
    "territori",
    "any",
    "tipus_de_territori",
]

def merge_dataset(
    master,
    dataset,
):

    return pd.merge(
        master,
        dataset,
        on=MERGE_KEYS,
        how="left",
    )


In [101]:
# ==================================================
# Validació inicial
# ==================================================

print("Validant datasets...")

for name, dataset in {

    "rent": df_rent,
    "income": df_income,
    "nat": df_nat,
    "pop": df_pop,
    "atur": df_atur,
    "est": df_est,
    "cont": df_cont,
    "sup": df_sup,
    "hut": df_hut,

}.items():

    print(f"{name}: {dataset.shape}")

Validant datasets...
rent: (1898, 4)
income: (511, 6)
nat: (8410, 5)
pop: (2117, 4)
atur: (1628, 4)
est: (2044, 5)
cont: (1898, 4)
sup: (1898, 4)
hut: (715, 4)


In [103]:
# ==================================================
# Merge inicial
# ==================================================

print("Construint dataset mestre...")

df_master = pd.merge(
    df_income,
    df_rent,
    on=MERGE_KEYS,
    how="outer",
    suffixes=("", "_rent"),
)

Construint dataset mestre...


In [105]:
# ==================================================
# Eliminar duplicats de variables
# ==================================================

if "preu_lloguer_rent" in df_master.columns:

    df_master["preu_lloguer"] = (
        df_master["preu_lloguer"]
        .fillna(df_master["preu_lloguer_rent"])
    )

    df_master = df_master.drop(
        columns=["preu_lloguer_rent"]
    )

In [107]:
# ==================================================
# Afegir població total
# ==================================================

df_master = merge_dataset(
    df_master,
    df_pop,
)

In [109]:
# ==================================================
# Nacionalitat LONG → WIDE
# ==================================================

nat_col = next(
    (
        c for c in df_nat.columns
        if "nacionalitat" in c
    ),
    None
)

if nat_col is None:

    raise ValueError(
        "No s'ha detectat "
        "la columna de nacionalitat."
    )

df_nat_wide = df_nat.pivot_table(
    index=MERGE_KEYS,
    columns=nat_col,
    values="poblacio",
    aggfunc="sum",
).reset_index()

# Validació duplicats
duplicates_nat = df_nat_wide.duplicated(
    subset=MERGE_KEYS
).sum()

print(
    "Duplicats nacionalitat:",
    duplicates_nat,
)

df_master = merge_dataset(
    df_master,
    df_nat_wide,
)

Duplicats nacionalitat: 0


In [111]:
# ==================================================
# Estudis LONG → WIDE
# ==================================================

tit_col = next(
    (
        c for c in df_est.columns
        if (
            "titul" in c
            or "estudi" in c
            or "escolar" in c
        )
    ),
    None
)

if tit_col is None:

    raise ValueError(
        "No s'ha detectat "
        "la columna de nivell educatiu."
    )

df_est_wide = df_est.pivot_table(
    index=MERGE_KEYS,
    columns=tit_col,
    values="poblacio",
    aggfunc="sum",
).reset_index()

# Validació duplicats
duplicates_est = df_est_wide.duplicated(
    subset=MERGE_KEYS
).sum()

print(
    "Duplicats estudis:",
    duplicates_est,
)

df_master = merge_dataset(
    df_master,
    df_est_wide,
)

Duplicats estudis: 0


In [113]:
# ==================================================
# Afegir variables addicionals
# ==================================================

additional_datasets = [

    df_atur,
    df_cont,
    df_sup,
    df_hut,
]

for dataset in additional_datasets:

    df_master = merge_dataset(
        df_master,
        dataset,
    )

In [115]:
# ==================================================
# Eliminar variables residuals
# ==================================================

cols_to_drop = [
    "areaid",
    "type",
]

df_master = df_master.drop(
    columns=cols_to_drop,
    errors="ignore",
)

In [117]:
# ==================================================
# Crear ID únic territorial
# ==================================================

df_master["territori_id"] = (
    df_master["territori"]
    .astype("category")
    .cat.codes
)

In [119]:
# ==================================================
# Ordenació final
# ==================================================

df_master = (
    df_master
    .sort_values(["territori", "any"])
    .reset_index(drop=True)
)

In [121]:
# ==================================================
# Validació final
# ==================================================

print("Shape final:")

print(df_master.shape)

print("\nDuplicats:")

duplicates = df_master.duplicated(
    subset=["territori", "any"]
).sum()

print(duplicates)

print("\nMissing values:")

display(
    df_master
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nPreview dataset:")

display(
    df_master.head()
)

Shape final:
(1898, 19)

Duplicats:
0

Missing values:


Primera etapa d’educació secundària i similar    1533
Segona etapa d’educació secundària i similar     1533
Educació primària o inferior                     1533
Educació superior                                1533
renda                                            1387
hut                                              1194
preu_lloguer                                     1032
superficie_mitja                                 1023
num_contractes                                   1022
atur                                              803
No consta                                          52
any                                                 0
Resta del món                                       0
tipus_de_territori                                  0
Resta de la Unió Europea                            0
Espanya                                             0
poblacio_total                                      0
territori                                           0
territori_id                


Preview dataset:


,any,tipus_de_territori,territori,renda,preu_lloguer,poblacio_total,Espanya,No consta,Resta de la Unió Europea,Resta del món,Educació primària o inferior,Educació superior,Primera etapa d’educació secundària i similar,Segona etapa d’educació secundària i similar,atur,num_contractes,superficie_mitja,hut,territori_id
0,2000,Barri,baro de viver,NaN,NaN,2368,2351.0,0.0,0.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2001,Barri,baro de viver,NaN,NaN,2373,2354.0,0.0,0.0,18.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,2002,Barri,baro de viver,NaN,NaN,2393,2363.0,0.0,0.0,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2003,Barri,baro de viver,NaN,NaN,2382,2334.0,0.0,6.0,41.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2004,Barri,baro de viver,NaN,NaN,2365,2301.0,0.0,0.0,60.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [123]:
# ==================================================
# Exportació
# ==================================================

save_csv(
    df_master,
    DATA_PROCESSED / "master.csv",
)

print(
    "MASTER DATAFRAME "
    "CREAT CORRECTAMENT"
)

MASTER DATAFRAME CREAT CORRECTAMENT
